In [70]:
from datetime import datetime
from multiprocess import Pool
import pandas
import psutil
import requests
from tqdm.notebook import tqdm

In [71]:
titled_players_df = pandas.read_csv("titled.csv")

In [72]:
usernames: list[str] = list(titled_players_df['username'])

In [73]:
def chunks(xs, n):
    n = max(1, n)
    return (xs[i : i + n] for i in range(0, len(xs), n))

In [74]:
user_agent = "ArithmeticErrorMozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:127.0) Gecko/20100101 Firefox/127.0"

In [75]:
def get_stats(username):
    try:
        url = f"https://api.chess.com/pub/player/{username}/stats"
        response = requests.get(url, headers={"User-Agent": user_agent})
        data: dict = response.json()
        rapid: str = "chess_rapid"
        blitz: str = "chess_blitz"
        bullet: str = "chess_bullet"
        # Rapid
        rapid_rating_deviation: int = data.get(rapid, {}).get("last", {}).get("rd", 0)
        rapid_rating_last: int = data.get(rapid, {}).get("last", {}).get("rating", 0)
        rapid_rating_best: int = data.get(rapid, {}).get("best", {}).get("rating", 0)
        # Blitz
        blitz_rating_deviation: int = data.get(blitz, {}).get("last", {}).get("rd", 0)
        blitz_rating_last: int = data.get(blitz, {}).get("last", {}).get("rating", 0)
        blitz_rating_best: int = data.get(blitz, {}).get("best", {}).get("rating", 0)
        # Bullet
        bullet_rating_deviation: int = data.get(bullet, {}).get("last", {}).get("rd", 0)
        bullet_rating_last: int = data.get(bullet, {}).get("last", {}).get("rating", 0)
        bullet_rating_best: int = data.get(bullet, {}).get("best", {}).get("rating", 0)
        return {
            "rapid_rating_last": rapid_rating_last,
            "rapid_rating_deviation": rapid_rating_deviation,
            "rapid_rating_best": rapid_rating_best,
            "blitz_rating_last": blitz_rating_last,
            "blitz_rating_deviation": blitz_rating_deviation,
            "blitz_rating_best": blitz_rating_best,
            "bullet_rating_last": bullet_rating_last,
            "bullet_rating_deviation": bullet_rating_deviation,
            "bullet_rating_best": bullet_rating_best,
        }
    except:
        return {
            "rapid_rating_deviation": 0,
            "rapid_rating_last": 0,
            "rapid_rating_best": 0,
            "blitz_rating_deviation": 0,
            "blitz_rating_last": 0,
            "blitz_rating_best": 0,
            "bullet_rating_deviation": 0,
            "bullet_rating_last": 0,
            "bullet_rating_best": 0,
        }

In [76]:
def get_player(username: str):
    try:

        url = f"https://api.chess.com/pub/player/{username}"
        response = requests.get(url, headers={"User-Agent": user_agent})
        data: dict = response.json()
        player_id: int = data.get("player_id", 0)
        name: str = data.get("name", "")
        title: str = data.get("title", "")
        # Time
        format_string = "%Y-%m-%d"
        # format_string = "%Y-%m-%d %H:%M:%S"
        joined: int = data.get("joined", 0)
        joined_date: str = datetime.fromtimestamp(joined).strftime(format_string)
        last_online: int = data.get("last_online", 0)
        last_online_date: str = datetime.fromtimestamp(last_online).strftime(format_string)
        # Stats
        stats = get_stats(username)
        # Country
        country_url = data.get("country", "")
        country_response = requests.get(country_url, headers={"User-Agent": user_agent})
        country_data = country_response.json()
        country: list[str] = country_data.get("name", "")
        # Streamer
        is_streamer: list[str] = data.get("is_streamer", False)
        twitch_url: list[str] = data.get("twitch_url", "")
        # Data
        return {
            "id": player_id,
            "username": username,
            "name": name,
            "title": title,
            "country": country,
            **stats,
            "is_streamer": is_streamer,
            "twitch_url": twitch_url,
            "joined": joined,
            "joined_date": joined_date,
            "last_online": last_online,
            "last_online_date": last_online_date,
        }
    except:
        return {
            "id": 0,
            "username": username,
            "name": "",
            "title": "",
            "country": "",
            **stats,
            "is_streamer": False,
            "twitch_url": "",
            "joined": 0,
            "joined_date": datetime.fromtimestamp(0).strftime(format_string),
            "last_online": 0,
            "last_online_date": datetime.fromtimestamp(0).strftime(format_string),
        }

In [77]:
def get_players(usernames: list[str]):
    pool = Pool(processes=3)
    players = pool.map(get_player, usernames)
    return players

In [78]:
psutil.cpu_count()

8

In [79]:
threads_count = psutil.cpu_count() // psutil.cpu_count(logical=False)
threads_count

1

In [80]:
players = []
chunk_usernames = list(chunks(usernames, 8))
for usernames in tqdm(chunk_usernames):
    chunk_players = get_players(usernames)
    players = players + chunk_players
    players_df = pandas.DataFrame(players)
    players_df.to_csv("titled-players.csv", index=False)

  0%|          | 0/1566 [00:00<?, ?it/s]

UnboundLocalError: cannot access local variable 'stats' where it is not associated with a value